# Clean Dataset

--------------------------------------------------------------

### Initializing the work environment

```bash
python -m venv venv
source venv/bin/activate
pip install -r requirement.txt
```

### Import the necessary libs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

-------------------------------------------------------------

The dataset consists of 3 columns: ID, a secret color, and the testimony. The color will be used to check if our clusters are accurate at the end of the learning phase of our project.

Our objective is to group similar testimonies into clusters, where each cluster potentially corresponds to a specific diagnosis or underlying disease.

In [ ]:
def init(path):
    df = pd.read_csv(path, header=None, names=[0, 1, 2])
    print(f"dataset load perfectly : {df.shape}")
    return df

In [ ]:
# function for print the dataset (debug or visualization purpose)
def print_df(n, df):
    df_print = df.head(n)
    print(f"this is the {n} first lines of the dataset:\n\n{df_print}")

## Clean the Dataset

---------------------------------------------

In this part of the project, we cannot modify the core nature of our dataset (unsupervised learning must be used with unlabeled data). However, we can analyze the testimonies and pre-process them by correcting spelling errors or removing irrelevant words that would make the learning harder.

The first issue I noticed is that the color column sometimes contains the testimony, leaving the column right after it with a 'missing value'. I will fix this issue by moving the text that accidentally fell into the color column back to the third column whenever this happens.

In [ ]:
def move_text_in_right_col(df):
    line_broke = df[2].isna()
    df.loc[line_broke, 2] = df.loc[line_broke, 1]
    df.loc[line_broke, 1] = '0x000000'
    return df

The next step of our data cleaning mission is to convert the messages to lowercase and remove all punctuation marks (these changes make it easier for the machine learning model).

In [ ]:
def simplify_text(df):
    df[2] = df[2].str.strip().str.lower()
    df[2] = df[2].str.replace(r'[^a-z\s]', ' ', regex=True)
    df[2] = df[2].str.replace(r'\s+', ' ', regex=True).str.strip()
    return df

Below is the function call used to preprocess the dataset and generate a clean version.

In [ ]:
df = init("./Student_Dataset.csv")
df = move_text_in_right_col(df)
df = simplify_text(df)
df.to_csv("cleaned_dataset.csv", index=False)

# Training with cleaned dataset

-------------------------------------------

## Init the dataset

In [ ]:
# init the dataset with Pandas
def init(path):
    df = pd.read_csv(path, header=None, names=[0, 1, 2])
    print(f"dataset load perfectly : {df.shape}")
    return df

df = init("./cleaned_dataset.csv")

-----------------------------------------

## Dataset analysis

This function will show us the most used words, and afterward, we can analyze the results to know what to do with our dataset.

In [ ]:
def most_writen_words(df, num_words=40):
    word_array = df[2].str.split()
    all_words = word_array.explode()
    frequences = all_words.value_counts().head(num_words)

    plt.figure(figsize=(12, 6))
    plt.bar(frequences.index, frequences.values, color='skyblue') 
    plt.title(f'Top {num_words} of the most words in the dataset')
    plt.xlabel('words')
    plt.ylabel('frequences')
    plt.xticks(rotation=45)
    plt.show()

most_writen_words(df)

We can see that the most written words are useless for the training because the model wants words with a real meaning, like a location or a feeling. We don't need words like 'I', 'my', 'and', etc.

-----------------------------------------------

## Words analysis

Fortunately, Scikit-learn provides a function that removes all the useless words (stop words) and converts the text into numerical values, allowing the model to understand what each line is about.

In [ ]:
# This function will vectorize, meaning it will put the words we are truly interested in into a matrix and assign them a value based on their presumed importance in the dataset
def vectorize_text(df):
    # To ensure our training goes as smoothly as possible, we will remove all unimportant words, such as isolated letters that were left behind when I replaced all punctuation with spaces during the cleaning step
    fantomes = ["s", "t", "m", "ve", "d", "ll", "re", "don", "didn", "doesn"]
    # And then, words that do not directly relate to the ailments, in order to leave only words with a real medical meaning
    storytelling = ["feel", "feels", "feeling", "felt", "notice", "noticed", "think", "thought", "believe", "start", "started", "keep", "kept", "got", "get", "went", "go", "came", "come", "make", "made", "like", "just"]
    # This line creates a list of all the words to remove, including the lists we defined above and a pre-made list of uninteresting English stopwords for machine learning programs
    stop_words_finaux = list(ENGLISH_STOP_WORDS) + fantomes + storytelling

    # This line configures the rules to define the importance of the words so that the training can proceed correctly
    vectorizer = TfidfVectorizer(stop_words=stop_words_finaux)
    # This line will create the matrix and transform the text into values according to the TF-IDF protocol
    matrice_tfidf = vectorizer.fit_transform(df[2])
    
    # This line will be used to debug and check the words that were kept
    mots_gardes = vectorizer.get_feature_names_out()
    # Visualizing the words kept by our code
    print(f"Number of highly relevant medical words kept: {len(mots_gardes)}")

    return matrice_tfidf, vectorizer

X, vectoriseur = vectorize_text(df)

Okay, now we are going to show the useful words kept by the function and try to understand how to find the different clusters.

Exemple extremement simplifier de la matrice X :

|  | Col n0 : "ache" | Col n1 : "chest" |
|:-------- |:--------:| --------:|
| Patient 1| 0.0   | 0.2    |
| Patient 2| 0.4   | 0.1    |

In [ ]:
# X: TF-IDF matrix, vectoriseur: The fitted TfidfVectorizer object (our vocabulary/dictionary)
def plot_top_tfidf_words(X, vectoriseur, top_n=20):

    # This line calculates the sum of all values in each column (i.e., the total TF-IDF score for each word across all documents)
    somme_tfidf = np.array(X.sum(axis=0)).flatten()
    # This line retrieves the actual readable words (feature names) corresponding to the matrix columns
    mots = vectoriseur.get_feature_names_out()
    # This sorts the array and extracts the indices of the top 20 words with the highest overall scores (sorted from lowest to highest among the top N)
    indices_top = somme_tfidf.argsort()[-top_n:]
    
    # This extracts the actual words and their corresponding scores using the sorted indices
    mots_top = [mots[i] for i in indices_top]
    scores_top = somme_tfidf[indices_top]
    
    plt.figure(figsize=(10, 8))
    plt.barh(mots_top, scores_top, color='coral')
    plt.title(f"Top {top_n} most 'discriminant' words")
    plt.xlabel("Score")
    plt.ylabel("Medical words")
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

plot_top_tfidf_words(X, vectoriseur)

Memo english word to french :
- sore = Endolori (qui fait mal)
- ache = Douleur
- cough = Toux
- sharp = Aigu / Vif

Interpretation of the "most discriminative words":

We have some body parts, which will be very useful to give a diagnosis. The body parts we have are: chest, skin, ear, and throat.
We also have some feelings (symptoms) about them: pain (ache), cold, sharp, long, sore, and cough (cough is not a feeling, but a good clue to give a diagnosis).
So, we can now try to understand and think about a way to train an AI to give a diagnosis based on a testimony. For example: if the testimony has "cough" and "throat" in it, it could be tonsillitis (maux de gorge).

---------------------------------

## Cloud Analysis and cluster reflection

Now, we will see how to group the different lines of our dataset into clusters and determine the optimal number of clusters we need.

Here, just after the learning program find out the best cluster. We reduced the dimensionality of our data to 2 parameters (2D) to plot it on a graph. However, as we can see, it is hard to tell which points belong together. To fix this, we will color the points into different clusters based on their similarities.

In [ ]:
def apply_kmeans_and_plot(X, n_clusters):
    # I chose K-Means as the machine learning algorithm because it is highly efficient and widely considered the industry standard for unsupervised learning and clustering.
    # This line is crucial: it initializes the model with a specific random seed (random_state) to ensure our results are reproducible and do not change with every run.
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    # This line trains our machine learning model on the entire TF-IDF word matrix, allowing it to learn the patterns and find similarities between patients.
    kmeans.fit(X)
    # After training, the model assigns a label to each patient, effectively grouping them into distinct clusters.
    labels_predict = kmeans.labels_
    
    # This sets up t-SNE, which will reduce the dimensionality of our data so it can be displayed on a 2D graph. I chose t-SNE because it is excellent at visually preserving local cluster groupings.
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    # This actively performs the dimensionality reduction, transforming our high-dimensional data into two highly representative 2D coordinates.
    X_tsne = tsne.fit_transform(X.toarray()) 
    
    plt.figure(figsize=(10, 8))
    # 1. We draw the background zones (the contour map) to show the cluster territories
    plt.tricontourf(X_tsne[:, 0], X_tsne[:, 1], labels_predict, cmap='tab10', alpha=0.2)
    # 2. We draw the patients (the data points) on top of the map
    plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_predict, cmap='tab10', alpha=1.0, edgecolors='w', s=50)
    plt.title("Archipelago of Diseases (t-SNE Reduction)")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()
    
    return kmeans

# We call the function. For now, we test with 5, but we will use the Elbow Method to find the optimal number.
apply_kmeans_and_plot(X, n_clusters=5)

As we can observe, points of the same color are mostly grouped in the same area, forming distinct clusters. However, the yellow points are widely spread out. This indicates we need a better way to separate the data into clusters. To achieve this, highly efficient techniques exist, such as the Elbow Method.

--------------------------------

The Elbow Method helps us find the perfect number of clusters. We test the algorithm with 1, 2, 3... up to 12 clusters and calculate the "inertia" (which measures how tight our groups are). Then, we plot these values on a graph.

The curve looks like an arm. The best number of clusters is exactly at the "elbow", where the line stops dropping fast and becomes almost flat. Adding more clusters after this point doesn't improve the result.

In [ ]:
# max_k=10 is a default value I chose because it was highly probable that the optimal number of clusters would fall between 1 and 10 (which turned out to be the case).
def find_optimal_k(X, max_k=10):
    # This list will store the inertia score calculated for each k in order to plot them on the graph later.
    inertias = []

    K_range = range(1, max_k + 1)
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        kmeans.fit(X)
        # This appends the inertia score. (Inertia is essentially the sum of squared distances between all data points in a cluster and their centroid; therefore, the lower the number, the tighter and better the clusters).
        inertias.append(kmeans.inertia_)
    
    plt.figure(figsize=(10, 6))
    plt.plot(K_range, inertias, marker='o', linestyle='-', color='teal', markersize=8)
    plt.title("Elbow Method to find the optimal K")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Inertia (Intra-cluster distance)")
    plt.xticks(K_range) # To clearly display all the numbers on the x-axis
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

find_optimal_k(X, max_k=12)

To interpret the Elbow Method graph, we must identify a significant change in the slope of the curve. Here, we can observe a very clear 'elbow' at K=9 (before the line becomes flat at 10), which gives us the optimal number of clusters for our dataset. It is worth pointing out that this dataset was clearly custom-made for this exercise: the shape of the curve is literally perfect, making the ideal number of clusters incredibly obvious to discover.

As we can observe, we do not simply select the lowest possible inertia. We must find a balance between a low inertia and a reasonable number of clusters. For instance, creating 1,000 clusters for 1,000 patients would result in an inertia of 0, but this would lead to overfitting and defeat the whole purpose of clustering.

we can now retry with 9 cluster :

In [ ]:
# We retrieve the trained K-Means model with our 9 optimal clusters
kmeans = apply_kmeans_and_plot(X, n_clusters=9)

def display_top_words_per_cluster(kmeans, vectorizer, n_mots=5):
    # We retrieve the model's "centers of gravity" (centroids) and our vocabulary/dictionary
    centroids = kmeans.cluster_centers_
    words = vectorizer.get_feature_names_out()
    
    # For each cluster, we identify the words with the highest TF-IDF scores
    for i, center in enumerate(centroids):
        # We sort the scores, extract the indices of the top 'n_mots', and reverse the array ([::-1]) to display them in descending order
        top_indices = center.argsort()[-n_mots:][::-1]
        top_words = [words[ind] for ind in top_indices]
        print(f"Cluster {i}: {', '.join(top_words)}")

# We call the function to display the top 6 defining words for each disease cluster
display_top_words_per_cluster(kmeans, vectoriseur, n_mots=6)

This choice is much more pertinent. Even if some points appear very far from the others, we must remember that we reduced many dimensions (parameters) to project these points onto a 2D graph, which distorts the real distances. Finally for a better understanding of what our clusters are made of we have a list of the most important words by  cluster and we can understand that all cluster is a teestimony for a specific deases. 